# Titanic 머신러닝 총정리 + 앙상블 기초

승객 정보로 생존을 예측하며 머신러닝 전체 흐름 연결

문제 정의 → 데이터 확인 → 행/열 → 특성(feature)·정답(target) → 데이터 준비 → 학습·확인·최종 test → 모델 학습(fit) → 예측(predict) → 평가(evaluate) → 비교 → 설정 조정 → 다음 전략

진행: 설명 → 짧은 판단 → 준비 셀 실행 → 결과 해석  
최종 test는 사용하지 않음


## 1. Titanic 표 읽기

- 행(row): 승객 1명
- 열(column): 승객 정보 1종류
- `Survived`: 생존 1, 비생존 0
- 승객 정보로 0/1을 맞히는 분류(classification) 문제

생존과 관련 있어 보이는 열 2개 선택
- 이유 한 줄 작성
- 아직 가설임


### 준비 1/2 — 기본 도구와 사용할 열

표를 읽는 데 필요한 도구와 열 이름 준비


In [31]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42
FEATURES = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
NUMERIC = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
CATEGORICAL = ["Sex", "Embarked"]

### 준비 2/2 — 데이터 나누기

학습·비교용 712행 준비  
최종 test 179행은 사용하지 않음


In [32]:
def locate_data():
    matches = sorted(Path.cwd().rglob("titanic_train.csv"))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"titanic_train.csv를 하나만 찾을 수 있어야 합니다. 현재 {len(matches)}개입니다."
        )
    return matches[0]
data = pd.read_csv(locate_data()).sort_values("PassengerId").reset_index(drop=True)
positions = np.arange(len(data))
development_positions, final_positions = train_test_split(
    positions, test_size=0.20, stratify=data["Survived"], random_state=SEED
)
development = data.iloc[development_positions].copy().reset_index(drop=True)
FINAL_HOLDOUT_ROWS = len(final_positions)
del final_positions

X = development[FEATURES].copy()
y = development["Survived"].astype(int).copy()

In [33]:
data[["PassengerId"] + FEATURES + ["Survived"]].head(3)


,PassengerId,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Survived
0,1,3,male,22.0,1,0,7.2500,S,0
1,2,1,female,38.0,1,0,71.2833,C,1
2,3,3,female,26.0,0,0,7.9250,S,1


### 특성(feature)과 정답(target)

| 열 | 뜻 |
|---|---|
| Pclass | 객실 등급 |
| Sex | 성별 |
| Age | 나이 |
| SibSp | 함께 탄 형제·자매·배우자 수 |
| Parch | 함께 탄 부모·자녀 수 |
| Fare | 운임 |
| Embarked | 승선 항구 |
| Survived | 맞힐 정답 |

- 특성 묶음: `X`
- 정답: `y`
- `Survived`를 X에 넣으면 정답을 미리 보게 됨


In [34]:
print("X shape:", X.shape, "y shape:", y.shape)
display(X.head(2))
display(y.head(2).rename("target: Survived"))


X shape: (712, 7) y shape: (712,)


,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,3,male,NaN,0,0,56.4958,S
1,2,male,NaN,0,0,0.0000,S


0    1
1    0
Name: target: Survived, dtype: int64

## 2. 데이터 준비

모델 계산 전 해결할 문제

1. 빈 `Age` 채우기
2. `female`, `S` 같은 글자 바꾸기
3. 큰 단위인 `Fare`가 거리 계산을 지배하지 않게 하기


In [35]:
display(X.isna().sum().rename("결측치 수"))

Pclass        0
Sex           0
Age         137
SibSp         0
Parch         0
Fare          0
Embarked      2
Name: 결측치 수, dtype: int64

### 원-핫 인코딩(one-hot encoding)

범주마다 0/1 열 생성

- 해당 범주: 1
- 다른 범주: 0
- 가짜 순위를 만들지 않음

실행 전 예상: 행과 열 중 무엇이 늘어날까?


In [36]:
from sklearn.preprocessing import OneHotEncoder

encoding_before = (
    X[["Sex", "Embarked"]]
    .dropna()
    .drop_duplicates()
    .sort_values(["Sex", "Embarked"])
    .reset_index(drop=True)
)
encoding_demo = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoded_values = encoding_demo.fit_transform(encoding_before)
encoding_after = pd.DataFrame(
    encoded_values,
    columns=encoding_demo.get_feature_names_out(["Sex", "Embarked"]),
)
display(encoding_before)
display(encoding_after.astype(int))
print("행 수 전/후:", len(encoding_before), len(encoding_after))

,Sex,Embarked
0,female,C
1,female,Q
2,female,S
3,male,C
4,male,Q
5,male,S


,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
0,1,0,1,0,0
1,1,0,0,1,0
2,1,0,0,0,1
3,0,1,1,0,0
4,0,1,0,1,0
5,0,1,0,0,1


행 수 전/후: 6 6


### 스케일링(scaling): 거리에서 큰 단위의 영향 줄이기

Passenger 540과 664 비교

- 나이 차이와 운임 차이 확인
- 원래 단위에서 거리를 더 크게 차지할 특성 예상
- 스케일링 전후 결과 비교

모델 점수나 모델 선택에는 사용하지 않는 원리 확인임


In [37]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# 원리 확인용: 학습·비교용 데이터 100행으로 기준을 만들고 두 승객을 변환
# 모델 점수나 설정 선택에는 사용하지 않음
distance_reference = X.iloc[:100][["Age", "Fare"]]
distance_pair = development.loc[
    development["PassengerId"].isin([540, 664]), ["PassengerId", "Age", "Fare"]
].sort_values("PassengerId")

distance_imputer = SimpleImputer(strategy="median")
distance_scaler = StandardScaler()
reference_ready = distance_imputer.fit_transform(distance_reference)
distance_scaler.fit(reference_ready)
pair_ready = distance_imputer.transform(distance_pair[["Age", "Fare"]])
pair_scaled = distance_scaler.transform(pair_ready)

raw_delta = np.abs(pair_ready[0] - pair_ready[1])
scaled_delta = np.abs(pair_scaled[0] - pair_scaled[1])
distance_comparison = pd.DataFrame({
    "특성": ["Age", "Fare"],
    "스케일링 전 차이": raw_delta,
    "스케일링 전 거리 몫": raw_delta ** 2 / np.sum(raw_delta ** 2),
    "스케일링 후 차이": scaled_delta,
    "스케일링 후 거리 몫": scaled_delta ** 2 / np.sum(scaled_delta ** 2),
})
display(distance_pair)
display(distance_comparison.style.format({
    "스케일링 전 차이": "{:.3f}", "스케일링 전 거리 몫": "{:.1%}",
    "스케일링 후 차이": "{:.3f}", "스케일링 후 거리 몫": "{:.1%}",
}))
print("전체 거리(전/후):", round(float(np.linalg.norm(raw_delta)), 3),
      round(float(np.linalg.norm(scaled_delta)), 3))


,PassengerId,Age,Fare
434,540,22.0,49.5000
222,664,36.0,7.4958


,특성,스케일링 전 차이,스케일링 전 거리 몫,스케일링 후 차이,스케일링 후 거리 몫
0,Age,14.000,10.0%,1.096,63.6%
1,Fare,42.004,90.0%,0.828,36.4%


전체 거리(전/후): 44.276 1.374


### 결과 해석

- 원래 단위에서 거리를 더 크게 차지한 특성 확인
- 스케일링 뒤 `Age`와 `Fare`의 거리 몫 비교
- KNN은 거리로 이웃을 찾으므로 스케일링이 중요함
- LR도 숫자 크기를 맞춰 학습 계산을 안정적으로 진행
- Tree는 한 열의 순서와 경계를 보므로 오늘은 스케일링하지 않음

모든 모델에 같은 전처리를 쓰는 것은 아님


### Pipeline — 데이터 준비와 모델을 한 흐름으로 묶기

각 CV 반복에서:

1. 학습 조각으로 빈칸 채우기·인코딩·스케일링 기준 학습
2. 확인 조각에는 같은 기준으로 변환만 적용
3. 변환된 데이터로 예측

전체 데이터를 먼저 가공하면 정보 누수가 생길 수 있음


In [38]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

def preprocessing(scale_numeric=False):
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))
    return ColumnTransformer([
        ("num", Pipeline(numeric_steps), NUMERIC),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), CATEGORICAL),
    ])

def model_pipe(model, scale_numeric=False):
    return Pipeline([("prepare", preprocessing(scale_numeric)), ("model", model)])

## 3. 학습·확인·최종 test 데이터

- 학습(train): 규칙을 배우는 데이터
- 확인(validation): 모델과 설정을 고르는 데이터
- 최종 test: 선택이 끝난 뒤 한 번 확인하는 데이터

같은 데이터로 배우고 채점하면 새 데이터 성능을 알기 어려움  
아래 실습은 학습·비교용 데이터 안에서 진행함


### fit → predict → evaluate

Logistic Regression 예시로 네 단계 확인

모델 준비 → 학습(fit) → 예측(predict) → 평가(evaluate)


In [39]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

def make_lr():
    return model_pipe(LogisticRegression(max_iter=2000, random_state=SEED), True)

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED
)
model = make_lr()
model.fit(X_train, y_train)
pred = model.predict(X_valid)
score = accuracy_score(y_valid, pred)
print("예측:", pred[:10])
print("실제:", y_valid.to_numpy()[:10])
print("정확도:", score)


예측: [0 0 1 0 0 0 0 1 0 0]
실제: [0 0 1 0 0 0 0 1 0 0]
정확도: 0.797752808988764


- `fit`: 학습 데이터에서 규칙 학습
- `predict`: 새 행의 0/1 예측
- `evaluate`: 실제 정답과 예측 비교


## 4. 5겹 교차검증(CV)과 기본 모델 비교

한 번만 나누면 쉬운 승객이나 어려운 승객이 우연히 몰릴 수 있음

- 데이터를 5조각으로 나눔
- 학습·확인 역할을 바꾸며 5번 반복
- 각 행은 자신을 학습에 쓰지 않은 모델에게 한 번 예측받음
- 이 예측을 OOF 예측이라 함

비교할 모델

- LR: 특성의 가중 합을 예측 확률로 바꿈
- KNN: 가까운 이웃의 정답을 봄
- Tree: 질문과 경계를 따라 마지막 잎에서 결정

실행 전 세 모델의 순위 예상


In [40]:
fold_role_table = pd.DataFrame({
    "반복": [1, 2, 3, 4, 5],
    "확인 조각": [1, 2, 3, 4, 5],
    "학습 조각": ["2·3·4·5", "1·3·4·5", "1·2·4·5", "1·2·3·5", "1·2·3·4"],
})
fold_role_table


,반복,확인 조각,학습 조각
0,1,1,2·3·4·5
1,2,2,1·3·4·5
2,3,3,1·2·4·5
3,4,4,1·2·3·5
4,5,5,1·2·3·4


In [41]:
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

lr = make_lr()
knn = model_pipe(KNeighborsClassifier(n_neighbors=11), True)
tree = model_pipe(DecisionTreeClassifier(max_depth=3, min_samples_leaf=5, random_state=SEED))
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def cv_row(name, candidate):
    scores = cross_val_score(candidate, X, y, cv=cv, scoring="accuracy", n_jobs=1)
    oof = cross_val_predict(candidate, X, y, cv=cv, method="predict", n_jobs=1)
    return {"모델/전략": name, "맞힌 수": int((oof == y).sum()),
            "CV 평균": scores.mean(), "CV 표준편차": scores.std()}

In [42]:
baseline_table = pd.DataFrame([
    cv_row("Logistic Regression", lr),
    cv_row("KNN (k=11)", knn),
    cv_row("Decision Tree (depth=3)", tree),
])
baseline_table.style.format({"CV 평균": "{:.3f}", "CV 표준편차": "{:.3f}"})


,모델/전략,맞힌 수,CV 평균,CV 표준편차
0,Logistic Regression,567,0.796,0.028
1,KNN (k=11),572,0.803,0.008
2,Decision Tree (depth=3),573,0.805,0.011


한 번의 높은 점수로 보편적인 승자를 정하지 않음

함께 볼 것: 평균, 흔들림, 오류 종류, 계산·설명 부담, 새 데이터 확인


## 5. 세 가지 비교와 모델 설정

모델 설정(hyperparameter): 학습 전에 사람이 정하는 값

- 모델 종류 비교: LR vs KNN vs Tree
- 같은 모델의 설정 비교: Tree 깊이 2 vs 3 vs 5
- 전략 비교: Tree vs Bagging·RF·Voting

Tree 깊이를 직접 바꾼 뒤 자동 비교가 필요한 이유 확인


In [43]:
manual_depth = []
for depth in [2, 3, 5]:
    candidate = model_pipe(DecisionTreeClassifier(
        max_depth=depth, min_samples_leaf=5, random_state=SEED
    ))
    manual_depth.append({"max_depth": depth, "CV 평균": cross_val_score(candidate, X, y, cv=cv).mean()})
pd.DataFrame(manual_depth).style.format({"CV 평균": "{:.3f}"})


,max_depth,CV 평균
0,2,0.791
1,3,0.805
2,5,0.809


### GridSearchCV — 정해 둔 설정 후보 자동 비교

- 주어진 후보만 같은 CV 기준으로 비교
- 후보 밖의 설정은 알 수 없음
- 최종 test는 사용하지 않음


In [44]:
from sklearn.model_selection import GridSearchCV

tree_search = GridSearchCV(
    model_pipe(DecisionTreeClassifier(random_state=SEED)),
    {"model__max_depth": [2, 3, 5], "model__min_samples_leaf": [1, 5, 10]},
    scoring="accuracy", cv=cv, n_jobs=1, refit=True,
).fit(X, y)
selection_result = pd.DataFrame([{
    "표의 역할": "설정 선택에 사용한 결과",
    "후보 수": 9,
    "선택 설정": str(tree_search.best_params_),
    "선택 CV 평균": tree_search.best_score_,
}])
selection_result.style.format({"선택 CV 평균": "{:.3f}"})


,표의 역할,후보 수,선택 설정,선택 CV 평균
0,설정 선택에 사용한 결과,9,"{'model__max_depth': 5, 'model__min_samples_leaf': 5}",0.809


설정 선택에 사용한 CV 결과임  
새 데이터 성능을 뜻하지 않음  
기본 모델 점수표와 합쳐 순위를 매기지 않음


## 6. Passenger 19의 Tree 판단 경로

한 승객이 어떤 질문과 경계를 지나 예측에 도착했는지 확인  
실행 전 처음 나올 특성 예상


In [45]:
p19_index = development.index[development["PassengerId"].eq(19)][0]
path_rows = []
for fold_number, (train_rows, valid_rows) in enumerate(cv.split(X, y), start=1):
    if p19_index not in valid_rows:
        continue
    p19_model = clone(tree).fit(X.iloc[train_rows], y.iloc[train_rows])
    p19_x = X.iloc[[p19_index]]
    transformed = p19_model.named_steps["prepare"].transform(p19_x)
    fitted_tree = p19_model.named_steps["model"]
    feature_names = p19_model.named_steps["prepare"].get_feature_names_out()
    leaf = int(fitted_tree.apply(transformed)[0])
    for node_id in fitted_tree.decision_path(transformed).indices:
        if int(node_id) == leaf:
            counts = fitted_tree.tree_.value[node_id][0]
            path_rows.append({"순서": len(path_rows)+1, "판단": "마지막 잎",
                              "승객 값": "-", "비교": "-", "기준": "-",
                              "마지막 잎의 0/1 비율": str(np.round(counts, 3).tolist())})
            continue
        feature_index = int(fitted_tree.tree_.feature[node_id])
        value = float(transformed[0, feature_index])
        threshold = float(fitted_tree.tree_.threshold[node_id])
        path_rows.append({"순서": len(path_rows)+1,
                          "판단": feature_names[feature_index],
                          "승객 값": round(value, 4),
                          "비교": "≤" if value <= threshold else ">",
                          "기준": round(threshold, 4), "마지막 잎의 0/1 비율": ""})
    p19_prediction = int(p19_model.predict(p19_x)[0])

display(development.loc[[p19_index], ["PassengerId"] + FEATURES + ["Survived"]])
display(pd.DataFrame(path_rows))
print("Passenger 19 OOF 예측/실제 정답:", p19_prediction, int(y.iloc[p19_index]))


,PassengerId,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Survived
9,19,3,female,31.0,1,0,18.0,S,0


,순서,판단,승객 값,비교,기준,마지막 잎의 0/1 비율
0,1,cat__Sex_female,1.0,>,0.5,
1,2,num__Pclass,3.0,>,2.5,
2,3,num__Fare,18.0,≤,22.9042,
3,4,마지막 잎,-,-,-,"[0.397, 0.603]"


Passenger 19 OOF 예측/실제 정답: 1 0


## 7. Tree의 흔들림 → Bagging 질문

Tree는 초반 경계가 바뀌면 뒤 질문도 함께 달라질 수 있음  
데이터가 조금 바뀔 때 예측이 흔들리는 성질을 분산(variance)이 크다고 표현함

같은 설정의 Tree 2개 비교

- 학습 표본 550행 중 491행이 같음
- 같은 100명에게 예측
- 결과가 모두 같을지 예상

성능 경쟁이 아니라 흔들림을 보는 고정 시연임


In [46]:
# 성능 비교가 아닌 원리 확인용 고정 시연
# 결과를 본 뒤 난수 시작점이나 표본 크기를 바꾸지 않음
instability_pool = np.arange(612)
sample_a = np.random.default_rng(101).choice(instability_pool, 550, replace=False)
sample_b = np.random.default_rng(202).choice(instability_pool, 550, replace=False)
probe_rows = np.arange(612, 712)

def make_instability_tree():
    return model_pipe(DecisionTreeClassifier(
        max_depth=None, min_samples_leaf=5, random_state=SEED
    ))

tree_a = make_instability_tree().fit(X.iloc[sample_a], y.iloc[sample_a])
tree_b = make_instability_tree().fit(X.iloc[sample_b], y.iloc[sample_b])
probe_pred_a = tree_a.predict(X.iloc[probe_rows])
probe_pred_b = tree_b.predict(X.iloc[probe_rows])
instability_disagreement = probe_pred_a != probe_pred_b
print("두 학습 표본의 공통 행:", len(set(sample_a) & set(sample_b)), "/ 550")
print("같은 100명에서 다른 예측:", int(instability_disagreement.sum()))
display(pd.DataFrame({
    "PassengerId": development.iloc[probe_rows]["PassengerId"].to_numpy(),
    "Tree A": probe_pred_a,
    "Tree B": probe_pred_b,
}).loc[instability_disagreement].head())


두 학습 표본의 공통 행: 491 / 550
같은 100명에서 다른 예측: 10


,PassengerId,Tree A,Tree B
0,391,0,1
3,173,0,1
18,31,1,0
42,225,0,1
43,241,1,0


### Bagging으로 이어지는 질문

서로 다른 학습 표본으로 여러 Tree를 만들고 판단을 모으면 Tree 하나의 흔들림을 줄일 수 있을까?

가능성을 만드는 방법이며 자동 개선은 아님


## 8. 앙상블(Ensemble) 지도

여러 모델의 판단을 합치는 전략

- 서로 다른 표본으로 같은 종류의 Tree 여러 개: Bagging
- Bagging + 분기마다 일부 특성만 보기: Random Forest
- 서로 다른 종류의 완성 모델 결합: Voting
- 앞 모델의 부족한 방향을 다음 모델이 순서대로 보정: Boosting

모델이 다르다는 사실만으로 성능이 좋아지는 것은 아님


### 10명 복원추출(bootstrap)

뽑은 사람을 다시 넣고 또 뽑는 방식

- 같은 사람이 여러 번 나올 수 있음
- 한 번도 나오지 않는 사람도 생김
- 표본이 달라지면 Tree의 경계와 오류도 달라질 수 있음


In [47]:
rng_a = np.random.default_rng(SEED)
rng_b = np.random.default_rng(SEED + 1)
bootstrap_a = rng_a.choice(np.arange(1, 11), size=10, replace=True)
bootstrap_b = rng_b.choice(np.arange(1, 11), size=10, replace=True)
display(pd.DataFrame({"표본 A": bootstrap_a, "표본 B": bootstrap_b}))
print("A에서 빠진 사람:", sorted(set(range(1, 11)) - set(bootstrap_a)))
print("B에서 빠진 사람:", sorted(set(range(1, 11)) - set(bootstrap_b)))


,표본 A,표본 B
0,1,6
1,8,7
2,7,5
3,5,1
4,5,6
5,9,1
6,1,3
7,7,9
8,3,5
9,1,6


A에서 빠진 사람: [2, 4, 6, 10]
B에서 빠진 사람: [2, 4, 8, 10]


In [48]:
from sklearn.ensemble import BaggingClassifier

bagging = model_pipe(BaggingClassifier(
    estimator=DecisionTreeClassifier(max_depth=3, min_samples_leaf=5, random_state=SEED),
    n_estimators=200, random_state=SEED, n_jobs=1,
))

In [49]:
def bagging_vote_detail():
    oof_pred = np.empty(len(X), dtype=int)
    vote_one = np.empty(len(X), dtype=int)
    vote_zero = np.empty(len(X), dtype=int)
    for train_idx, valid_idx in cv.split(X, y):
        fitted = clone(bagging).fit(X.iloc[train_idx], y.iloc[train_idx])
        oof_pred[valid_idx] = fitted.predict(X.iloc[valid_idx])
        prepared = fitted.named_steps["prepare"].transform(X.iloc[valid_idx])
        estimators = fitted.named_steps["model"].estimators_
        subsets = fitted.named_steps["model"].estimators_features_
        votes = np.vstack([
            est.predict(prepared[:, subset]) for est, subset in zip(estimators, subsets)
        ])
        vote_one[valid_idx] = votes.sum(axis=0).astype(int)
        vote_zero[valid_idx] = len(estimators) - vote_one[valid_idx]
    return oof_pred, vote_one, vote_zero

bag_pred, bag_vote_one, bag_vote_zero = bagging_vote_detail()
tree_oof_for_bag = cross_val_predict(tree, X, y, cv=cv, method="predict", n_jobs=1)
tree_only_correct = (tree_oof_for_bag == y.to_numpy()) & (bag_pred != y.to_numpy())
bag_only_correct = (bag_pred == y.to_numpy()) & (tree_oof_for_bag != y.to_numpy())


### Bagging 안의 판단 확인

200개 Tree의 최종 0/1 다수결과 예측 확률 평균 결과가 다를 수 있음  
실행 전 가능한 이유 예상


In [50]:
near = np.where((bag_vote_zero == 102) & (bag_vote_one == 98))[0]
print("0:1 표가 102:98인 승객:", development.loc[near, "PassengerId"].tolist())
print("Bagging 최종 예측:", bag_pred[near].tolist())
for pid in [329, 255]:
    i = development.index[development["PassengerId"].eq(pid)][0]
    print(pid, "실제/Bagging/1표/0표:",
          int(y.iloc[i]), int(bag_pred[i]), int(bag_vote_one[i]), int(bag_vote_zero[i]))


0:1 표가 102:98인 승객: [817]
Bagging 최종 예측: [1]
329 실제/Bagging/1표/0표: 1 0 91 109
255 실제/Bagging/1표/0표: 0 0 81 119


In [51]:
print("Tree만 정답:", int(tree_only_correct.sum()))
print("Bagging만 정답:", int(bag_only_correct.sum()))
print("순변화:", int(bag_only_correct.sum() - tree_only_correct.sum()))


Tree만 정답: 6
Bagging만 정답: 6
순변화: 0


새로 맞힌 수와 새로 틀린 수를 함께 확인  
두 방향의 차이로 순변화 계산


## 9. Random Forest와 Hard/Soft Voting

- Random Forest: 표본을 바꾸고, 분기마다 볼 특성도 일부만 선택
- Hard Voting: 각 모델의 최종 0/1을 한 표씩 셈
- Soft Voting: 각 모델의 예측 확률을 평균냄

Soft Voting은 예측 확률이 믿을 만한지도 중요함


In [52]:
from sklearn.ensemble import RandomForestClassifier, VotingClassifier

rf = model_pipe(RandomForestClassifier(
    n_estimators=200, max_depth=3, min_samples_leaf=5, max_features="sqrt",
    random_state=SEED, n_jobs=1,
))
hard_voting = VotingClassifier(
    estimators=[("lr", clone(lr)), ("knn", clone(knn)), ("tree", clone(tree))], voting="hard"
)
soft_voting = VotingClassifier(
    estimators=[("lr", clone(lr)), ("knn", clone(knn)), ("tree", clone(tree))], voting="soft"
)

### Hard Voting — 최종 0/1을 한 표씩 세기

`1, 0, 1`이면 1이 두 표임  
실행 전 최종 예측 선택


In [53]:
hard_labels = np.array([1, 0, 1])
print("모델의 최종 0/1:", hard_labels.tolist())
print("Hard Voting 결과:", int(hard_labels.mean() >= 0.5))


모델의 최종 0/1: [1, 0, 1]
Hard Voting 결과: 1


### Soft Voting — 1일 예측 확률 평균

`0.51, 0.10, 0.55`의 평균이 0.5를 넘을지 먼저 선택


In [54]:
soft_probabilities = np.array([0.51, 0.10, 0.55])
soft_mean = float(soft_probabilities.mean())
print("모델의 1일 예측 확률:", soft_probabilities.tolist())
print("평균:", round(soft_mean, 3))
print("Soft Voting 결과:", int(soft_mean >= 0.5))


모델의 1일 예측 확률: [0.51, 0.1, 0.55]
평균: 0.387
Soft Voting 결과: 0


### 다수결이 고친 판단과 망친 판단

Passenger 19와 856에서 세 모델의 예측 확인  
Hard Voting 계산 후 실제 정답과 비교


In [55]:
hard_oof = cross_val_predict(hard_voting, X, y, cv=cv)
member_pred = np.vstack([
    cross_val_predict(lr, X, y, cv=cv),
    cross_val_predict(tree, X, y, cv=cv),
    cross_val_predict(knn, X, y, cv=cv),
])

voting_case_rows = []
for pid in [19, 856]:
    i = development.index[development["PassengerId"].eq(pid)][0]
    voting_case_rows.append({
        "PassengerId": pid,
        "LR": int(member_pred[0, i]),
        "Tree": int(member_pred[1, i]),
        "KNN-11": int(member_pred[2, i]),
    })
voting_case_votes = pd.DataFrame(voting_case_rows)
voting_case_votes

,PassengerId,LR,Tree,KNN-11
0,19,0,1,0
1,856,1,0,0


두 행의 0/1을 한 표씩 세어 Hard Voting 계산  
아직 실제 정답은 공개하지 않음


In [56]:
voting_case_reveal = voting_case_votes.copy()
voting_case_reveal["Hard Voting"] = (
    voting_case_reveal[["LR", "Tree", "KNN-11"]].sum(axis=1) >= 2
).astype(int)
voting_case_reveal["실제 정답"] = [
    int(y.iloc[development.index[development["PassengerId"].eq(pid)][0]])
    for pid in voting_case_reveal["PassengerId"]
]
voting_case_reveal["결과"] = np.where(
    voting_case_reveal["Hard Voting"].eq(voting_case_reveal["실제 정답"]),
    "정답", "오답",
)
voting_case_reveal

,PassengerId,LR,Tree,KNN-11,Hard Voting,실제 정답,결과
0,19,0,1,0,0,0,정답
1,856,1,0,0,0,1,오답


### 두 사례를 전체 전략 비교와 연결

도움이 된 변화와 해가 된 변화를 모두 확인한 뒤 전체 결과 비교


In [57]:
ensemble_table = pd.DataFrame([
    cv_row("Tree", tree),
    cv_row("Bagging", bagging),
    cv_row("Random Forest", rf),
    cv_row("Hard Voting", hard_voting),
    cv_row("Soft Voting", soft_voting),
])
ensemble_table.style.format({"CV 평균": "{:.3f}", "CV 표준편차": "{:.3f}"})

,모델/전략,맞힌 수,CV 평균,CV 표준편차
0,Tree,573,0.805,0.011
1,Bagging,573,0.805,0.014
2,Random Forest,573,0.805,0.029
3,Hard Voting,573,0.805,0.019
4,Soft Voting,574,0.806,0.026


두 사례와 전체 결과를 한 문장으로 정리

`Voting은 ___을 모으지만, 개선 여부는 ___으로 확인해야 함.`

한 번의 가장 큰 숫자로 보편적인 승자를 정하지 않음


## 10. Session 1 정리

1. 전체 머신러닝 흐름을 순서대로 설명
2. KNN·LR과 Tree의 전처리가 다른 이유 설명
3. CV와 GridSearchCV의 역할 구분
4. Tree 흔들림이 Bagging으로 이어지는 이유와 앙상블이 자동 개선이 아닌 이유 설명

다음 질문: 완성 모델을 나란히 합치는 대신, 앞 예측의 부족한 방향을 다음 모델이 순서대로 고치면 어떻게 될까?
